[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aloshdenny/lbs-agentic-ai/blob/main/notebooks/agentic-orchestration.ipynb)

# A Very Simple Crew

Chapter 02 named CrewAI as one of the frameworks in the wild: agents organised into a "crew," each with a role, a goal, and a task, collaborating like a small team with a manager splitting up the work.

This notebook is the smallest version of that idea: two agents, two tasks, one crew. A **Researcher** gathers a few facts, then a **Writer** turns them into a short brief. CrewAI passes the Researcher's output to the Writer automatically, that hand-off is the framework doing for you what Chapter 03 built by hand.

In [ ]:
%pip install -q crewai litellm

## Reusable: the LLM

One LLM config, defined once, same Groq model as the other notebook. Every agent below shares this same object.

One patch before that: CrewAI sends a prompt-caching hint that Groq's API doesn't accept yet, which throws an error before your agents get a chance to run. This turns that hint into a no-op so requests reach Groq clean. Safe to ignore, and safe to delete once CrewAI's Groq support catches up.

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
import crewai.llms.cache as _crewai_cache

_crewai_cache.mark_cache_breakpoint = lambda message: message  # see note above

GROQ_API_KEY = ""
MODEL = "groq/openai/gpt-oss-20b"  # crewAI/litellm want the "groq/" prefix on the model name

import os
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

llm = LLM(model=MODEL, temperature=0.4)

## Reusable: the two agents

A role, a goal, and a backstory each. No tools this time, just the crew mechanic itself, kept deliberately small.

In [ ]:
researcher = Agent(
    role="Researcher",
    goal="Find clear, accurate facts about the given topic",
    backstory="You are a careful researcher who sticks to verifiable facts and keeps answers short.",
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Writer",
    goal="Turn research notes into a short, friendly brief",
    backstory="You write concise two-paragraph briefs for a general audience, no jargon.",
    llm=llm,
    verbose=True,
)

## The crew: tasks + kickoff

Each task belongs to one agent. `write_task` names `research_task` as context, so CrewAI feeds the Researcher's output into the Writer's prompt automatically, no manual message-passing.

`Process.sequential` just means: run the tasks in order. CrewAI also has `Process.hierarchical`, where a manager agent decides the order itself, worth a look once this simple version makes sense.

In [ ]:
research_task = Task(
    description="Research the topic: {topic}. List 3-4 key facts.",
    expected_output="A short bullet list of 3-4 key facts about the topic.",
    agent=researcher,
)

write_task = Task(
    description="Using the research notes, write a two-paragraph brief about {topic} for a college workshop audience.",
    expected_output="A two-paragraph brief, friendly tone, no jargon.",
    agent=writer,
    context=[research_task],
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True,
)

## Try it

CrewAI prints each agent's thinking as it works. Scroll to the bottom for the finished brief.

One more Jupyter-specific note: a notebook's kernel already has an asyncio event loop running, and CrewAI's plain `crew.kickoff()` refuses to start inside one (it'll raise a `RuntimeError` telling you exactly this). Use `crew.kickoff_async()` with `await` instead, Jupyter cells support top-level `await` natively, so no other change is needed. A plain `.py` script wouldn't hit this and could use `crew.kickoff()` directly.

In [ ]:
topic = input("Topic for the crew to research and write about: ")
result = await crew.kickoff_async(inputs={"topic": topic})

print("\n\n=== FINAL BRIEF ===")
print(result)